# VIGENT Sports 데이터 학습 — 요가 동작 '인식' 모델

**목적**: '이게 무슨 동작인가'(나무/전사/플랭크…)를 자동 인식하는 분류기 학습.
**방식**: 이미지가 아니라 **MediaPipe 포즈 키포인트(33점)**로 학습 → 가볍고 정확, 브라우저에서도 추론 가능.
**보완 관계**: '정답 각도'(규칙)는 *잘했나 채점*, 이 모델은 *무슨 동작인가 인식*. 둘이 합쳐짐.

### 시작 전
1. 런타임 → GPU 불필요(키포인트 분류는 가벼움, CPU로 충분)
2. 데이터: 동작별 이미지 폴더 (yoga/tree/*.jpg, yoga/warrior2/*.jpg …) 또는 공개 데이터셋(Yoga-82 등)


## 1) 설치


In [ ]:
!pip -q install mediapipe scikit-learn numpy


## 2) 데이터 준비
- 구글드라이브나 업로드로 `data/` 아래에 **동작별 폴더**로 이미지 배치:
  `data/mountain/*.jpg`, `data/tree/*.jpg`, `data/warrior2/*.jpg`, `data/plank/*.jpg`
- 공개 데이터셋(Yoga-82, Kaggle Yoga Poses)을 받아 같은 구조로 두면 됩니다.


In [ ]:
DATA_DIR='data'   # 동작별 하위폴더 구조


## 3) 이미지 → 포즈 키포인트 추출
각 이미지에서 MediaPipe 로 33점을 뽑아 '특징벡터'로 만듭니다(어깨중심 기준 정규화).


In [ ]:
import os, glob, numpy as np, cv2, mediapipe as mp
mp_pose = mp.solutions.pose
def to_features(img):
    with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose:
        res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if not res.pose_landmarks: return None
    lm = res.pose_landmarks.landmark
    pts = np.array([[p.x,p.y,p.z] for p in lm])
    # 어깨중심 기준 평행이동 + 어깨너비로 스케일 정규화(위치·크기 불변)
    c = (pts[11]+pts[12])/2; pts = pts - c
    s = np.linalg.norm(pts[11]-pts[12]) or 1.0
    return (pts/s).flatten()

X, y = [], []
labels = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(f'{DATA_DIR}/{d}')])
for li,label in enumerate(labels):
    for fp in glob.glob(f'{DATA_DIR}/{label}/*'):
        im = cv2.imread(fp);  feat = to_features(im) if im is not None else None
        if feat is not None: X.append(feat); y.append(li)
X=np.array(X); y=np.array(y)
print('동작:', labels, '· 샘플 수:', len(X))


## 4) 분류기 학습 + 정확도 측정(하니스)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
clf = RandomForestClassifier(n_estimators=300, random_state=42).fit(Xtr,ytr)
pred = clf.predict(Xte)
print('정확도:', round(accuracy_score(yte,pred)*100,1),'%')
print(classification_report(yte,pred,target_names=labels))


## 5) 모델 저장 + 다운로드
- 학습된 분류기와 라벨을 저장해 VIGENT 에 연결합니다.


In [ ]:
import joblib, json
joblib.dump(clf,'yoga_pose_clf.pkl')
json.dump(labels, open('yoga_labels.json','w'), ensure_ascii=False)
from google.colab import files
files.download('yoga_pose_clf.pkl'); files.download('yoga_labels.json')


## 6) 다음 (VIGENT 연결)
1. 받은 `yoga_pose_clf.pkl`·`yoga_labels.json` 을 `vigent-core/weights/` 로
2. Claude 에게 '요가 동작 인식 모델 연결해줘' 라고 하세요
3. sports 화면이 동작을 **자동 인식** → 그 동작의 '정답 각도'로 채점까지 연결

> 키포인트 분류라 가벼워서, 나중에 브라우저(TF.js)에서도 추론 가능.
